In [1]:
from pathlib import Path
from food_pipeline.spark_session import get_spark
from pyspark.sql import functions as F

ROOT = Path.cwd().parent
spark = get_spark()
RAW = (ROOT / "data" / "raw" / "online_food_delivery.csv").as_posix()

In [2]:
raw = spark.read.csv(RAW, header=True, inferSchema=False)
raw.printSchema()

root
 |-- Age: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Marital Status: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- Monthly Income: string (nullable = true)
 |-- Educational Qualifications: string (nullable = true)
 |-- Family size: string (nullable = true)
 |-- Customer Type: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- Pin code: string (nullable = true)
 |-- Output: string (nullable = true)
 |-- Feedback: string (nullable = true)
 |-- _c13: string (nullable = true)



In [3]:
print("rows:", raw.count(), "columns:", len(raw.columns))
raw.show(5, truncate=False)

rows: 388 columns: 14


+---+------+--------------+----------+--------------+--------------------------+-----------+-------------+--------+---------+--------+------+---------+----+
|Age|Gender|Marital Status|Occupation|Monthly Income|Educational Qualifications|Family size|Customer Type|latitude|longitude|Pin code|Output|Feedback |_c13|
+---+------+--------------+----------+--------------+--------------------------+-----------+-------------+--------+---------+--------+------+---------+----+
|20 |Female|Single        |Student   |No Income     |Post Graduate             |4          |Frequent     |12.9766 |77.5993  |560001  |Yes   |Positive |Yes |
|24 |Female|Single        |Student   |Below Rs.10000|Graduate                  |3          |Regular      |12.977  |77.5773  |560009  |Yes   |Positive |Yes |
|22 |Male  |Single        |Student   |Below Rs.10000|Post Graduate             |3          |Regular      |12.9551 |77.6593  |560017  |Yes   |Negative |Yes |
|22 |Female|Single        |Student   |No Income     |Gradu

In [4]:
raw.describe().show(truncate=False)

+-------+------------------+------+--------------+----------+--------------+--------------------------+------------------+-------------+--------------------+------------------+------------------+------+---------+----+
|summary|Age               |Gender|Marital Status|Occupation|Monthly Income|Educational Qualifications|Family size       |Customer Type|latitude            |longitude         |Pin code          |Output|Feedback |_c13|
+-------+------------------+------+--------------+----------+--------------+--------------------------+------------------+-------------+--------------------+------------------+------------------+------+---------+----+
|count  |388               |388   |388           |388       |388           |388                       |388               |388          |388                 |388               |388               |388   |388      |388 |
|mean   |24.628865979381445|NULL  |NULL          |NULL      |NULL          |NULL                      |3.2809278350515463|NULL  

### Finding: everything is a string
All columns were read as strings, so min/max are alphabetical, not numeric
(e.g. Monthly Income min = "10001 to 25000", max = "No Income").
Pin code has a "mean" of 560040, which is meaningless because a pin code is an ID, not a number.
No nulls: every column has count = 388.

**Decision:** cast age, family_size, latitude, longitude to numbers in Silver; keep pin_code as a string;
map income bands to an ordered number (income_band_order 0–4).

In [5]:
for c in ["Gender", "Marital Status", "Occupation", "Monthly Income",
          "Educational Qualifications", "Customer Type", "Output", "Feedback"]:
    raw.groupBy(c).count().orderBy("count", ascending=False).show(truncate=False)

+------+-----+
|Gender|count|
+------+-----+
|Male  |222  |
|Female|166  |
+------+-----+

+-----------------+-----+
|Marital Status   |count|
+-----------------+-----+
|Single           |268  |
|Married          |108  |
|Prefer not to say|12   |
+-----------------+-----+



+--------------+-----+
|Occupation    |count|
+--------------+-----+
|Student       |207  |
|Employee      |118  |
|Self Employeed|54   |
|House wife    |9    |
+--------------+-----+

+---------------+-----+
|Monthly Income |count|
+---------------+-----+
|No Income      |187  |
|25001 to 50000 |69   |
|More than 50000|62   |
|10001 to 25000 |45   |
|Below Rs.10000 |25   |
+---------------+-----+



+--------------------------+-----+
|Educational Qualifications|count|
+--------------------------+-----+
|Graduate                  |177  |
|Post Graduate             |174  |
|Ph.D                      |23   |
|School                    |12   |
|Uneducated                |2    |
+--------------------------+-----+

+-------------+-----+
|Customer Type|count|
+-------------+-----+
|Regular      |218  |
|Frequent     |146  |
|New          |24   |
+-------------+-----+



+------+-----+
|Output|count|
+------+-----+
|Yes   |301  |
|No    |87   |
+------+-----+

+---------+-----+
|Feedback |count|
+---------+-----+
|Positive |317  |
|Negative |71   |
+---------+-----+



### Finding: inconsistent categories and small groups
- Typos: "Self Employeed" (54 rows), "House wife" (9 rows).
- Very small groups: Uneducated = 2, School = 12, Prefer not to say = 12, New customers = 24.
- Output is imbalanced: 301 Yes vs 87 No.

**Decision:** standardise to "Self Employed" and "Homemaker" in Silver.
Always show sample size next to any rate in Gold.

In [6]:
raw.select(F.concat(F.lit("["), F.col("Feedback"), F.lit("]")).alias("feedback_bracketed")) \
   .groupBy("feedback_bracketed").count().show(truncate=False)

+------------------+-----+
|feedback_bracketed|count|
+------------------+-----+
|[Negative ]       |71   |
|[Positive]        |317  |
+------------------+-----+



### Finding: trailing whitespace
All 71 "Negative" values in Feedback are stored as "Negative " (with a trailing space).

**Impact:** a filter like feedback == "Negative" would match 0 rows.

**Decision:** trim() every string column in Silver.

In [7]:
total = raw.count()
unique = raw.dropDuplicates().count()
print(f"total={total}, unique={unique}, duplicates={total - unique}")

total=388, unique=285, duplicates=103


### Finding: exact duplicates
388 rows, 285 unique, so 103 exact duplicates (about 27%).
No customer ID, so we can't tell repeat submissions apart from different people with identical answers.

**Decision:** keep all rows in Bronze (raw record); deduplicate in Silver using a record_hash and document the trade-off.

In [8]:
mismatches = raw.filter(F.col("Output") != F.col("_c13")).count()
print("rows where Output and _c13 differ:", mismatches)

rows where Output and _c13 differ: 0


### Finding: unnamed 14th column
The CSV header ends with a trailing comma, so Spark created a column called _c13.
It equals Output in all 388 rows (0 mismatches).

**Decision:** keep it in Bronze as raw_extra; add a DQ rule that it matches Output, then drop it in Silver.

In [9]:
pin_check = (
    raw.groupBy("Pin code")
       .agg(F.countDistinct("latitude", "longitude").alias("n_locations"))
       .filter(F.col("n_locations") > 1)
)
print("pin codes with more than one location:", pin_check.count())
print("distinct pin codes:", raw.select("Pin code").distinct().count())

pin codes with more than one location: 0


distinct pin codes: 77


### Finding: location data is consistent
77 distinct pin codes, and each maps to exactly one latitude/longitude pair.

**Decision:** pin_code is a clean key for dim_location in Gold.

## Landing batches
The raw file is split into two simulated batches by `scripts/make_landing_batches.py`.
Reconciliation: the batch counts must add up to the raw count, with the same columns.

In [10]:
LANDING = (ROOT / "data" / "raw" / "landing").as_posix()

b1 = spark.read.csv(f"{LANDING}/batch_01.csv", header=True)
b2 = spark.read.csv(f"{LANDING}/batch_02.csv", header=True)
print("batch_01:", b1.count(), "batch_02:", b2.count(), "total:", b1.count() + b2.count())
print("same columns as raw:", b1.columns == raw.columns == b2.columns)

batch_01: 200 batch_02: 188 total: 388
same columns as raw: True
